# Compare FPJ and Chain-MS across datasets

This notebook compares the winning regions, per-game runtime, strategy size, and decision-diagram nodes for FPJ and Chain-MS. By default it loads the saved all-dataset results, so opening the notebook does not rerun the solvers. Set `RUN_FRESH = True` to benchmark the selected datasets again.

The tables are filtered by `DATASETS`: use `None` for all seven benchmark sets, one name such as `'Lily'`, or a list such as `['Lily', 'AMBA']`. Table 1 reports dataset graph statistics; Tables 2 and 3 compare runtime and diagram nodes. Chain-MS node counts are for its chain-transition BDD relations.


In [ ]:
from pathlib import Path
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import gc
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from parity_bdd_lab import (
    benchmark_dataset_names,
    benchmark_selection,
    collect_bdd_garbage,
    iter_benchmark_files,
    read_pg,
    records_to_dataframe,
)
from parity_bdd_lab.fpj import _Encoding

METHODS = ['FPJ', 'Chain-MS']
DATASETS = None  # None for all; or e.g. 'Lily' / ['Lily', 'AMBA']
RUN_FRESH = True
REPEATS = 1
WORKERS = 4
RESULTS_CSV = CODE / 'data' / 'benchmarks' / 'multistrategy_fpj_comparison.csv'


## Select or load benchmark results

The saved CSV contains one run for each method/game pair across all seven datasets. A fresh run reports progress as each game/method finishes and writes to a separate CSV so the saved all-dataset results remain available.


In [ ]:
available_datasets = benchmark_dataset_names()
dataset_lookup = {name.casefold(): name for name in available_datasets}
if DATASETS is None:
    ACTIVE_DATASETS = list(available_datasets)
elif isinstance(DATASETS, str):
    ACTIVE_DATASETS = [dataset_lookup[DATASETS.casefold()]]
else:
    ACTIVE_DATASETS = [dataset_lookup[name.casefold()] for name in DATASETS]

if RUN_FRESH:
    fresh_records = benchmark_selection(
        ACTIVE_DATASETS,
        methods=METHODS,
        repeats=REPEATS,
        workers=WORKERS,
        progress=True,
    )
    all_results = records_to_dataframe(fresh_records)
    if len(ACTIVE_DATASETS) == len(available_datasets):
        output_csv = RESULTS_CSV
    else:
        output_csv = CODE / 'data' / 'benchmarks' / 'multistrategy_fpj_comparison_selected.csv'
    all_results.to_csv(output_csv, index=False)
    print(f'Saved fresh results to {output_csv}')
else:
    if not RESULTS_CSV.is_file():
        raise FileNotFoundError(f'{RESULTS_CSV} not found; set RUN_FRESH = True')
    all_results = pd.read_csv(RESULTS_CSV)
    all_results = all_results[
        all_results['dataset'].isin(ACTIVE_DATASETS)
        & all_results['method'].isin(METHODS)
    ].copy()

all_results = all_results.sort_values(['dataset', 'game', 'method', 'run']).reset_index(drop=True)
print(f'{len(all_results)} run records across {all_results.game.nunique()} games and {all_results.dataset.nunique()} datasets')
display(all_results)


## Check that FPJ and Chain-MS agree

Both methods should report the same winning-region fingerprint for every game. The table below lists any missing or mismatched comparisons.


In [ ]:
fingerprints = all_results.pivot_table(
    index=['dataset', 'game'],
    columns='method',
    values='solution_fingerprint',
    aggfunc='first',
)
for method in METHODS:
    if method not in fingerprints.columns:
        fingerprints[method] = pd.NA
fingerprints['regions_match'] = (
    fingerprints['FPJ'].notna()
    & fingerprints['Chain-MS'].notna()
    & (fingerprints['FPJ'] == fingerprints['Chain-MS'])
)
print(f"Matching games: {int(fingerprints['regions_match'].sum())} / {len(fingerprints)}")
display(fingerprints[~fingerprints['regions_match']])


## Compare the actual allowed strategy edges

The saved benchmark CSV stores strategy edge counts, not edge lists, so this cell solves the selected games again and compares exact edge sets for each player. FPJ and Chain-MS are run independently. It also reports how many controlled vertices have more than one allowed move.


In [ ]:
from collections import Counter
from parity_bdd_lab import chain_multistrategy, fpj

strategy_rows = []
strategy_difference_rows = []
for dataset in ACTIVE_DATASETS:
    paths = tuple(iter_benchmark_files(dataset))
    for game_number, game_path in enumerate(paths, 1):
        game = read_pg(game_path)
        fpj_solution = fpj(game)
        chain_solution = chain_multistrategy(game)
        for player in (0, 1):
            fpj_edges = set(fpj_solution.even_strategy if player == 0 else fpj_solution.odd_strategy)
            chain_edges = set(chain_solution.even_strategy if player == 0 else chain_solution.odd_strategy)
            fpj_only = fpj_edges - chain_edges
            chain_only = chain_edges - fpj_edges
            owner_region = fpj_solution.even_wins if player == 0 else fpj_solution.odd_wins
            controlled = [v for v in owner_region if game.owner[v] == player]
            fpj_choices = Counter(source for source, _ in fpj_edges)
            chain_choices = Counter(source for source, _ in chain_edges)
            strategy_rows.append({
                'dataset': dataset, 'game': game_path.name, 'player': player,
                'FPJ allowed edges': len(fpj_edges),
                'Chain-MS allowed edges': len(chain_edges),
                'shared edges': len(fpj_edges & chain_edges),
                'FPJ only': len(fpj_only), 'Chain-MS only': len(chain_only),
                'identical': fpj_edges == chain_edges,
                'FPJ multi-choice vertices': sum(fpj_choices[v] > 1 for v in controlled),
                'Chain-MS multi-choice vertices': sum(chain_choices[v] > 1 for v in controlled),
            })
            if fpj_only or chain_only:
                strategy_difference_rows.append({
                    'dataset': dataset, 'game': game_path.name, 'player': player,
                    'FPJ-only edges': tuple(sorted(fpj_only, key=repr)),
                    'Chain-MS-only edges': tuple(sorted(chain_only, key=repr)),
                })
        if game_number % 10 == 0 or game_number == len(paths):
            print(f'{dataset}: compared {game_number}/{len(paths)} games')
        del chain_solution, fpj_solution, game
        gc.collect()

strategy_comparison = pd.DataFrame(strategy_rows)
strategy_differences = pd.DataFrame(strategy_difference_rows)
print(f"Identical player/game edge sets: {strategy_comparison['identical'].sum()} / {len(strategy_comparison)}")
display(strategy_comparison)
display(strategy_differences)


## Per-game comparison

Runtime is averaged across repeats for each game. `Chain-MS / FPJ` below 1 means Chain-MS ran faster; above 1 means it took longer.


In [ ]:
per_game = (
    all_results
    .groupby(['dataset', 'game', 'method'], as_index=False)
    .agg(
        mean_seconds=('runtime_seconds', 'mean'),
        even_winning_vertices=('even_winning_vertices', 'first'),
        odd_winning_vertices=('odd_winning_vertices', 'first'),
        strategy_edges=('strategy_edges', 'mean'),
        diagram_nodes=('bdd_nodes', 'mean'),
    )
)
per_game_comparison = per_game.pivot(
    index=['dataset', 'game'],
    columns='method',
    values=['mean_seconds', 'strategy_edges', 'diagram_nodes'],
)
per_game_comparison.columns = [f'{metric}__{method}' for metric, method in per_game_comparison.columns]
per_game_comparison = per_game_comparison.reset_index()
per_game_comparison['Chain-MS / FPJ time'] = (
    per_game_comparison['mean_seconds__Chain-MS'] / per_game_comparison['mean_seconds__FPJ']
)
display(per_game_comparison)


## Table 1: dataset statistics

The static BDD node count uses the project's game encoding and is independent of the solver results.


In [ ]:
game_statistics = []
for dataset in ACTIVE_DATASETS:
    for game_path in iter_benchmark_files(dataset):
        game = read_pg(game_path)
        priorities = tuple(game.priority[v] for v in game.vertices)
        vertex_count = len(game.vertices)
        edge_count = sum(len(game.successors[v]) for v in game.vertices)
        encoding = _Encoding(game)
        collect_bdd_garbage(encoding.bdd)
        game_statistics.append({
            'dataset': dataset,
            'game': game_path.name,
            'vertices': vertex_count,
            'edges': edge_count,
            'min priority': min(priorities),
            'max priority': max(priorities),
            'static BDD nodes': len(encoding.bdd),
        })
        del encoding, game
        gc.collect()

game_statistics = pd.DataFrame(game_statistics)
dataset_summary_rows = []
for dataset in ACTIVE_DATASETS:
    games = game_statistics[game_statistics['dataset'] == dataset]
    dataset_summary_rows.append({
        'Set': dataset,
        '#games': games['game'].nunique(),
        'avg. n': games['vertices'].mean(),
        'max. n': games['vertices'].max(),
        'priorities': f"{games['min priority'].min()}–{games['max priority'].max()}",
        'avg. out-degree': games['edges'].sum() / games['vertices'].sum(),
        'avg. #BDD nodes': games['static BDD nodes'].mean(),
    })
dataset_summary = pd.DataFrame(dataset_summary_rows)
display(dataset_summary.round({'avg. n': 0, 'avg. out-degree': 2, 'avg. #BDD nodes': 0}))


## Table 2: cumulative runtime by dataset

Runtime is summed over games within each repeat, then averaged across repeats.


In [ ]:
per_repeat_runtime = (
    all_results
    .groupby(['dataset', 'method', 'run'], as_index=False)['runtime_seconds']
    .sum()
)
runtime_by_dataset = (
    per_repeat_runtime
    .groupby(['dataset', 'method'])['runtime_seconds']
    .mean()
    .unstack('method')
    .reindex(index=ACTIVE_DATASETS, columns=METHODS)
    .rename(columns={method: method.casefold() for method in METHODS})
)
display(runtime_by_dataset.round(3))


## Table 3: average decision-diagram nodes by dataset

FPJ reports its BDD node count; Chain-MS reports the total nodes in its BDD relations over full-game and forced-strategy chains. These counts describe different encodings and should not be read as directly equivalent measures.


In [ ]:
diagram_nodes_by_dataset = (
    all_results
    .groupby(['dataset', 'method'])['bdd_nodes']
    .mean()
    .unstack('method')
    .reindex(index=ACTIVE_DATASETS, columns=METHODS)
    .rename(columns={method: method.casefold() for method in METHODS})
)
display(diagram_nodes_by_dataset.round(1))
